## KMS in practice — S3, EBS, RDS

Three integrations that come up constantly.

### S3 — four encryption modes

| Mode | Keys | Notes |
|---|---|---|
| **SSE-S3** | AWS-managed, no KMS | default; cheapest; no per-request audit |
| **SSE-KMS** | KMS key (managed or CMK) | every GET triggers a KMS `Decrypt`; CloudTrail-logged |
| **SSE-C** | customer-supplied per request | AWS encrypts/decrypts but never stores the key |
| **Client-side** | app encrypts before upload | AWS never sees plaintext at all |

The SSE-KMS gotcha: **every S3 GET calls KMS to decrypt**. At high request rates that hits the per-region KMS rate limit (5,500–30,000 req/sec). The fix is **S3 Bucket Keys** — S3 derives a short-lived bucket-level data key from KMS and reuses it across many objects, slashing KMS API calls. Enable bucket keys whenever you use SSE-KMS at scale.

### EBS

**EBS encryption** is chosen at volume creation (`aws/ebs` or a CMK). Encryption covers data at rest, snapshots, and disk I/O. You **cannot encrypt an existing unencrypted volume in place** — snapshot it, copy the snapshot with encryption enabled, restore. Turn on **account-level default encryption** so every new volume is encrypted automatically.

### RDS

**RDS encryption** is also set at creation and covers the instance, automated backups, snapshots, and read replicas. Like EBS, you can't encrypt an existing unencrypted instance directly — snapshot, copy the snapshot with encryption, restore. **TLS** protects data in transit to the database; KMS protects it at rest.

The through-line: KMS is chosen **at resource-creation time**, encrypts at rest transparently, and (for storage) is very hard to add after the fact.